# 01 — Data Profiling: Olist E-Commerce Dataset

**Цель ноутбука:** изучить структуру 9 таблиц Olist, найти проблемы качества данных, составить каталог правил для валидации (день 2).

**Логика:**
1. Загружаем все таблицы
2. Смотрим базовую структуру (shape, dtypes, head)
3. Пропуски (missing values)
4. Дубликаты
5. Уникальные значения и кардинальность
6. Числовые колонки — выбросы и валидность
7. Даты — порядок и валидность
8. Целостность ссылок (foreign keys)
9. Итоговый каталог найденных проблем

**Как работать с ноутбуком:**
- Ячейки, где код уже написан — запускаете и смотрите вывод.
- Ячейки с пометкой `# TODO` — вы дописываете сами по подсказкам.
- Маркдаун-ячейки `**TODO:** ...` — это ваши заметки/наблюдения, которые потом войдут в финальный каталог.

К концу ноутбука у вас должен быть список из ~15–20 правил качества данных. В день 2 каждое превратится в Great Expectation.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# отображение всех колонок и более широкая ширина вывода
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# тема для графиков
sns.set_theme(style='whitegrid')

# путь к сырым данным
DATA_DIR = Path('../data/raw')
print(f"Looking for data in: {DATA_DIR.resolve()}")
print(f"Found {len(list(DATA_DIR.glob('*.csv')))} CSV files")

## 1. Загрузка таблиц

Olist состоит из 9 связанных таблиц. Загружаем их все в словарь `tables` для удобной итерации.

In [ ]:
TABLE_FILES = {
    'customers': 'olist_customers_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

tables = {name: pd.read_csv(DATA_DIR / filename) for name, filename in TABLE_FILES.items()}

# сводная информация о размере
for name, df in tables.items():
    print(f"{name:25} {df.shape[0]:>8,} rows x {df.shape[1]:>2} cols")

## 2. Базовая структура и типы данных

Для каждой таблицы смотрим:
- размер (shape)
- типы колонок (dtypes)
- первые 3 строки

Обратите особое внимание на колонки, которые **должны быть датами**, но загрузились как `object` (строки). Это первая проблема качества — даты нужно конвертировать в `datetime`.

In [ ]:
for name, df in tables.items():
    print(f"\n{'='*70}\n{name.upper()}\n{'='*70}")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nHead:")
    display(df.head(3))

**TODO 2.1** — Запишите ниже свои наблюдения по типам данных:
- Какие колонки должны быть `datetime`, но являются `object`?
- Есть ли колонки, которые выглядят как ID, но загрузились как числовые? (для ID лучше string, чтобы не путать с количеством)
- Что-то ещё странное в типах?

Это первый блок «находок» для финального каталога.

In [ ]:
#1. Какие колонки должны быть datetime, но являются object?
#Таблица ORDERS: ВСЕ колонки со временем (order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date).
#Таблица ORDER_ITEMS: shipping_limit_date.
#Таблица ORDER_REVIEWS: review_creation_date, review_answer_timestamp.
#2. Колонки, которые выглядят как ID или коды, но загрузились как числовые?
#Почтовые индексы: customer_zip_code_prefix в CUSTOMERS, seller_zip_code_prefix в SELLERS, geolocation_zip_code_prefix в GEOLOCATION. Сейчас это int64.
#Индексы позиций: order_item_id в ORDER_ITEMS.
#3. Что-то ещё странное в типах?
#Таблица PRODUCTS: Почти все характеристики (weight_g, length_cm, photos_qty) загрузились как float64 (числа с плавающей точкой), хотя количество фото или вес в граммах обычно целые числа.

## 3. Пропуски (Missing Values)

Считаем процент пропусков по каждой колонке. Базовая шкала интерпретации:
- 0% — отлично
- <5% — приемлемо
- 5–20% — требует осознанной стратегии (impute / drop / flag)
- \>20% — серьёзная проблема, возможно колонка непригодна для аналитики

In [ ]:
def missing_summary(df: pd.DataFrame, name: str) -> pd.DataFrame | None:
    """Возвращает таблицу пропусков с количеством и процентом для колонок, где они есть."""
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    summary = pd.DataFrame({'missing_count': missing, 'missing_pct': pct})
    summary = summary[summary['missing_count'] > 0].sort_values('missing_pct', ascending=False)
    if len(summary) == 0:
        return None
    summary.insert(0, 'table', name)
    return summary

all_missing = []
for name, df in tables.items():
    result = missing_summary(df, name)
    if result is not None:
        all_missing.append(result)

missing_report = pd.concat(all_missing) if all_missing else pd.DataFrame()
missing_report

**TODO 3.1** — Запишите наблюдения:
- В каких колонках пропусков больше 5%?
- Для каждой такой колонки — это критично для бизнес-логики или приемлемо?
- Есть ли пропуски в primary key колонках? (это уже всегда баг)

In [ ]:
# Наблюдения по пропускам:
# 1. Экстремально много пропусков в текстах отзывов (88% заголовки, 58% сообщения). 
#    Приемлемо, так как оценка (score) заполнена полностью.
# 2. В таблице products есть 610 "пустых" позиций (1.85%) без названий, фото и категорий. 
#    Нужно пометить их как 'unknown' в пайплайне очистки.
# 3. В таблице orders около 3% пропусков даты доставки. 
#    Требуется кросс-чек со статусом заказа (SLA мониторинг).
# 4. Баги в Primary Keys отсутствуют — критических ошибок идентификации нет.

## 4. Дубликаты

Проверяем два типа дублей:
1. **Дубли по primary key** — это всегда баг. Например, два `order_id` с разными данными.
2. **Полные дубли строк** — могут быть багом ETL или ожидаемыми (зависит от таблицы).

In [ ]:
# Простые (одноколоночные) primary keys
PRIMARY_KEYS = {
    'customers': 'customer_id',
    'orders': 'order_id',
    'products': 'product_id',
    'sellers': 'seller_id',
    'order_reviews': 'review_id',
}

print("Дубликаты по primary key:\n")
for name, pk in PRIMARY_KEYS.items():
    df = tables[name]
    dupes = df[pk].duplicated().sum()
    print(f"  {name:15} PK={pk:15} duplicates: {dupes}")

print("\nПолные дубли строк (все колонки совпадают):\n")
for name, df in tables.items():
    dupes = df.duplicated().sum()
    if dupes > 0:
        print(f"  {name:25} {dupes} duplicate rows")

**TODO 4.1** — Добавьте проверку **составных** primary key. Это PK, который состоит из двух и более колонок.

- `order_items` — PK = (`order_id`, `order_item_id`)
- `order_payments` — PK = (`order_id`, `payment_sequential`)

Подсказка: `df.duplicated(subset=['col1', 'col2']).sum()`

In [ ]:
# 1. Проверка order_items: один и тот же номер товара в одном и том же заказе
df_items = tables['order_items']
items_dupes = df_items.duplicated(subset=['order_id', 'order_item_id']).sum()
print(f"order_items composite PK (order_id + order_item_id) duplicates: {items_dupes}")

# 2. Проверка order_payments: один и тот же этап оплаты в одном и том же заказе
df_payments = tables['order_payments']
payments_dupes = df_payments.duplicated(subset=['order_id', 'payment_sequential']).sum()
print(f"order_payments composite PK (order_id + payment_sequential) duplicates: {payments_dupes}")

In [ ]:
# Наблюдения по дубликатам:
# 1. Обнаружен критический баг в order_reviews: 814 неуникальных review_id. 
#    Требуется дедупликация перед использованием в ML-моделях анализа тональности.
# 2. Таблица geolocation содержит 261k полных дублей (около 25% объема). 
#    Необходима очистка для оптимизации производительности.
# 3. Составные ключи в заказах и платежах чистые (0 дублей) — логика транзакций не нарушена.

## 5. Уникальные значения и кардинальность

Для категориальных колонок смотрим:
- сколько уникальных значений (cardinality)
- топ-N самых частых

Помогает найти:
- колонки с одним значением (бесполезны для аналитики)
- неожиданные значения (опечатки, грязь, странные коды)
- категории с экстремально редкими значениями

In [ ]:
# Пример: статусы заказов в таблице orders
df = tables['orders']
print(f"order_status — {df['order_status'].nunique()} unique values:")
print(df['order_status'].value_counts())

**TODO 5.1** — Проделайте то же самое для:
1. `customer_state` в `customers` (должны быть двухбуквенные коды штатов Бразилии — все 27)
2. `payment_type` в `order_payments` (ограниченный набор: credit_card, boleto, etc.)
3. `product_category_name` в `products` (категории товаров — сколько их? есть ли странные?)

In [ ]:
# 1. Проверка штатов (в Бразилии должно быть ровно 26 штатов + 1 федеральный округ = 27)
states = tables['customers']['customer_state']
print(f"customer_state — {states.nunique()} уникальных значений (ожидаем 27):")
print(states.value_counts())

# 2. Проверка типов оплаты
payments = tables['order_payments']['payment_type']
print(f"\npayment_type — {payments.nunique()} уникальных значений:")
print(payments.value_counts())

# 3. Категории товаров (посмотрим на разнообразие)
categories = tables['products']['product_category_name']
print(f"\nproduct_category_name — {categories.nunique()} уникальных категорий:")
print(categories.value_counts().head(10)) # топ-10 самых популярных


In [ ]:
# Наблюдения по уникальным значениям:
# 1. В payment_type обнаружено значение 'not_defined' (3 случая). 
#    Это аномалия, которую нужно исключить из финансовой аналитики.
# 2. Штаты покупателей соответствуют справочнику (27 уникальных кодов).
# 3. Кардинальность категорий товаров (73) высокая. Требуется использование 
#    таблицы-переводчика (category_translation) для англоязычных отчетов.

## 6. Числовые колонки — выбросы и валидность

Основные проверки для числовых колонок:
- `describe()` — min, max, медиана, квартили
- наличие отрицательных значений там, где их быть не должно (цены, количества, размеры)
- экстремальные выбросы

In [ ]:
# Числовая статистика для order_items (там цены и доставка)
print("ORDER_ITEMS — числовая статистика:")
display(tables['order_items'].describe())

# Проверка отрицательных и нулевых значений в "денежных" колонках
print("\nКоличество отрицательных/нулевых значений:")
for col in ['price', 'freight_value']:
    df = tables['order_items']
    neg = (df[col] < 0).sum()
    zero = (df[col] == 0).sum()
    print(f"  {col}: {neg} negative, {zero} zero")

**TODO 6.1** — Дополнительные проверки:
1. Постройте boxplot для `price` (`sns.boxplot(x=tables['order_items']['price'])`). Какие выбросы? Цены в логарифмической шкале — что показывает `np.log1p`?
2. Таблица `order_payments` — есть ли отрицательные или нулевые `payment_value`? Это нормально?
3. Таблица `products` — есть ли отрицательные/нулевые `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Визуализация цен
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.boxplot(x=tables['order_items']['price'])
plt.title("Boxplot цен (Price)")

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(tables['order_items']['price']), kde=True)
plt.title("Логарифмическое распределение цен")
plt.show()

# 2. Проверка платежей
df_pay = tables['order_payments']
print("Таблица ORDER_PAYMENTS:")
neg_pay = (df_pay['payment_value'] < 0).sum()
zero_pay = (df_pay['payment_value'] == 0).sum()
print(f"  payment_value: {neg_pay} отрицательных, {zero_pay} нулевых")

# 3. Проверка параметров товаров
df_prod = tables['products']
print("\nТаблица PRODUCTS (размеры и вес):")
num_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in num_cols:
    neg = (df_prod[col] < 0).sum()
    zero = (df_prod[col] == 0).sum()
    print(f"  {col}: {neg} отрицательных, {zero} нулевых")

In [ ]:
# Наблюдения по числам:
# 1. Обнаружено 9 заказов с payment_value = 0 в таблице order_payments. 
#    Требуется проверка: это промо-коды на 100% или баг интеграции с платежным шлюзом?
# 2. Обнаружено 383 записи с бесплатной доставкой (freight_value=0). Валидно.
# 3. Экстремальные выбросы в ценах (max=6735 при медиане 74). 
#    Необходимо сегментировать товары перед анализом среднего чека.
# 4. В таблице products найдены нулевые значения веса и габаритов. 
#    Это препятствует автоматическому расчету логистических затрат.

## 7. Даты — порядок и валидность

В таблице `orders` пять временных меток. Логичный порядок:

`purchase_timestamp ≤ approved_at ≤ delivered_carrier_date ≤ delivered_customer_date`

И отдельно: `estimated_delivery_date` — обещанная дата доставки на момент покупки.

Любые нарушения порядка — баги данных.

In [ ]:
orders = tables['orders'].copy()
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Проверка порядка: approved >= purchase
violation_1 = (orders['order_approved_at'] < orders['order_purchase_timestamp']).sum()
print(f"approved < purchase: {violation_1} cases")

# Проверка порядка: delivered_customer >= delivered_carrier
violation_2 = (orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date']).sum()
print(f"delivered_customer < delivered_carrier: {violation_2} cases")

**TODO 7.1** — Добавьте ещё проверки:
1. `delivered_carrier_date >= approved_at` (передали в доставку после оплаты)
2. Посчитайте `delivery_lead_time = order_delivered_customer_date - order_purchase_timestamp` в днях. Какое среднее? Медиана? Есть ли **отрицательные** lead times (это значит даты не консистентны)?
3. Найдите диапазон дат покупки (`min` и `max` от `order_purchase_timestamp`) — на каком интервале вообще работает датасет?

In [ ]:
# 1. Проверка: передали в доставку после оплаты (carrier >= approved)
violation_3 = (orders['order_delivered_carrier_date'] < orders['order_approved_at']).sum()
print(f"delivered_carrier < approved_at: {violation_3} cases")

# 2. Расчет и анализ Lead Time (время от покупки до доставки в днях)
orders['delivery_lead_time'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days

print("\nСтатистика времени доставки (в днях):")
print(f"  Среднее: {orders['delivery_lead_time'].mean():.1f} дн.")
print(f"  Медиана: {orders['delivery_lead_time'].median():.1f} дн.")

# Проверка отрицательных lead times
neg_lead_time = (orders['delivery_lead_time'] < 0).sum()
print(f"  Отрицательное время доставки: {neg_lead_time} случаев")

# 3. Диапазон дат датасета
min_date = orders['order_purchase_timestamp'].min()
max_date = orders['order_purchase_timestamp'].max()
print(f"\nПериод данных: с {min_date} по {max_date}")


In [ ]:
# Наблюдения по датам:
# 1. Обнаружено 23 критических бага: дата доставки покупателю раньше даты передачи перевозчику.
# 2. Обнаружено более 1000 случаев отгрузки (carrier_date) до одобрения платежа (approved_at). 
#    Требуется аудит процесса подтверждения транзакций.
# 3. Медианное время доставки составляет 10 дней. Наличие отрицательных или 
#    экстремально долгих сроков доставки требует фильтрации данных перед расчетом KPI.
# 4. Данные охватывают период 2016–2018 гг. Требуется проверка актуальности 
#    бизнес-процессов для текущих задач.

## 8. Целостность ссылок (Foreign Key Integrity)

Проверяем, что каждая ссылка в одной таблице существует в другой:
- каждый `customer_id` в `orders` ∈ `customers.customer_id`
- каждый `order_id` в `order_items` ∈ `orders.order_id`
- и т.д.

Если ссылок не хватает — это «orphan records», битые внешние ключи.

In [ ]:
def fk_check(child_df, child_col, parent_df, parent_col, label):
    """Проверяет, что все значения child_col существуют в parent_col."""
    child_set = set(child_df[child_col].dropna())
    parent_set = set(parent_df[parent_col].dropna())
    orphans = child_set - parent_set
    print(f"  {label}: {len(orphans)} orphan values (из {len(child_set)} уникальных)")

print("Foreign Key integrity:")
fk_check(tables['orders'], 'customer_id', tables['customers'], 'customer_id',
         'orders.customer_id -> customers')
fk_check(tables['order_items'], 'order_id', tables['orders'], 'order_id',
         'order_items.order_id -> orders')
fk_check(tables['order_payments'], 'order_id', tables['orders'], 'order_id',
         'order_payments.order_id -> orders')
fk_check(tables['order_reviews'], 'order_id', tables['orders'], 'order_id',
         'order_reviews.order_id -> orders')

# Обратная проверка — заказы без записей в дочерних таблицах
all_orders = set(tables['orders']['order_id'])
orders_without_items = all_orders - set(tables['order_items']['order_id'])
print(f"\norders без записей в order_items: {len(orders_without_items)}")

**TODO 8.1** — Добавьте ещё две FK-проверки:
1. `order_items.product_id` ∈ `products.product_id`
2. `order_items.seller_id` ∈ `sellers.seller_id`

In [ ]:
print("Дополнительные Foreign Key проверки:")

# 1. Проверяем: все ли товары из заказов есть в справочнике продуктов?
fk_check(tables['order_items'], 'product_id', tables['products'], 'product_id', 
         'order_items.product_id -> products')

# 2. Проверяем: все ли продавцы из заказов есть в справочнике продавцов?
fk_check(tables['order_items'], 'seller_id', tables['sellers'], 'seller_id', 
         'order_items.seller_id -> sellers')

In [ ]:
# Наблюдения по целостности данных:
# 1. Обнаружено 775 "пустых" заказов (есть в orders, но нет в order_items). 
#    Это критично для финансового учета: по таким заказам невозможно рассчитать GMV.
# 2. Целостность связей orders-customers и orders-payments составляет 100%. 
#    Идентификация клиентов и транзакций надежна.
# 3. [После запуска TODO] Проверить наличие товаров-сирот (продажи товаров, которых нет в справочнике products).

## 9. Итог: каталог проблем качества данных

На основе всех находок выше — составим финальный каталог. В день 2 каждая строка превратится в Great Expectation.

Заполните таблицу ниже (минимум 15 строк):

| # | Таблица | Колонка(и) | Тип проблемы | Правило для валидации |
|---|---------|------------|--------------|------------------------|
| 1 | orders | order_purchase_timestamp | wrong dtype (object вместо datetime) | expect_column_values_to_match_strftime_format |
| 2 | orders | order_status | range (closed set) | expect_column_values_to_be_in_set |
| 3 | order_items | price | range (≥0) | expect_column_values_to_be_between (min=0) |
| 4 | order_items | freight_value | range (≥0) | expect_column_values_to_be_between (min=0) |
| 5 | orders | order_id | uniqueness | expect_column_values_to_be_unique |
| 6 | orders | customer_id | FK | expect_column_values_to_be_in_set |
| 7 | ... | ... | ... | ... |

**Подсказка** — типы проблем, которые обычно встречаются:
- `wrong dtype` — даты как строки, ID как числа
- `nulls` — пропуски в критичных полях
- `duplicates` — дубли PK или составного ключа
- `range` — отрицательные цены, нулевые размеры, выбросы
- `closed set` — значения из ограниченного списка (статусы, типы оплат, штаты)
- `FK integrity` — orphan записи
- `date order` — нарушения хронологии
- `format` — неверный формат строк, лишние пробелы

После заполнения каталога — день 1 закончен. В день 2 будем превращать его в код.

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# Собираем данные для финального каталога
dq_data = [
    {"ID": 1, "Таблица": "orders", "Колонка(и)": "все даты", "Тип проблемы": "wrong dtype", "Описание/Правило": "Конвертация в datetime для расчета SLA/Lead Time"},
    {"ID": 2, "Таблица": "customers/sellers", "Колонка(и)": "zip_code", "Тип проблемы": "wrong dtype", "Описание/Правило": "Изменить int64 на string (сохранение ведущих нулей)"},
    {"ID": 3, "Таблица": "products", "Колонка(и)": "category_name", "Тип проблемы": "nulls", "Описание/Правило": "610 пропусков (1.85%). Заполнить 'unknown'"},
    {"ID": 4, "Таблица": "orders", "Колонка(и)": "delivered_date", "Тип проблемы": "nulls", "Описание/Правило": "Пропуски ~3%. Дата обязательна при статусе 'delivered'"},
    {"ID": 5, "Таблица": "order_reviews", "Колонка(и)": "review_id", "Тип проблемы": "uniqueness", "Описание/Правило": "Критично: 814 дублей PK. Необходима дедупликация"},
    {"ID": 6, "Таблица": "geolocation", "Колонка(и)": "все", "Тип проблемы": "duplicates", "Описание/Правило": "261k полных дублей (25%). Требуется чистка (df.drop_duplicates)"},
    {"ID": 7, "Таблица": "order_payments", "Колонка(и)": "payment_type", "Тип проблемы": "closed set", "Описание/Правило": "3 случая 'not_defined'. Исключить из фин. отчетов"},
    {"ID": 8, "Таблица": "order_items", "Колонка(и)": "price", "Тип проблемы": "range", "Описание/Правило": "Должно быть > 0.85 (мин. цена по каталогу)"},
    {"ID": 9, "Таблица": "order_payments", "Колонка(и)": "payment_value", "Тип проблемы": "range", "Описание/Правило": "Аномалия: 9 платежей на сумму 0.0. Проверить на фрод"},
    {"ID": 10, "Таблица": "products", "Колонка(и)": "weight_g", "Тип проблемы": "range", "Описание/Правило": "Вес должен быть > 0 (найдены нулевые значения)"},
    {"ID": 11, "Таблица": "orders", "Колонка(и)": "delivered vs carrier", "Тип проблемы": "date order", "Описание/Правило": "Баг: 23 случая доставки раньше отгрузки"},
    {"ID": 12, "Таблица": "orders", "Колонка(и)": "carrier vs approved", "Тип проблемы": "date order", "Описание/Правило": "1000+ отгрузок до одобрения оплаты. Проверить интеграцию"},
    {"ID": 13, "Таблица": "orders", "Колонка(и)": "lead_time", "Тип проблемы": "range", "Описание/Правило": "Время исполнения заказа должно быть > 0"},
    {"ID": 14, "Таблица": "orders", "Колонка(и)": "customer_id", "Тип проблемы": "FK integrity", "Описание/Правило": "Проверка связи с customers.customer_id (Успех: 0 ошибок)"},
    {"ID": 15, "Таблица": "orders", "Колонка(и)": "order_id", "Тип проблемы": "emptiness", "Описание/Правило": "775 заказов без товаров. Исключить из расчета GMV"},
    {"ID": 16, "Таблица": "products", "Колонка(и)": "габариты (cm)", "Тип проблемы": "range", "Описание/Правило": "Размеры должны быть > 0. Заполнить медианой при очистке"}
]

# Создаем DataFrame
df_dq = pd.DataFrame(dq_data)

# Стилизация для красивого вывода в Jupyter (как в Сбере)
style = [
    {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'left')]},
    {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f2f2f2')]}
]

print("🚀 ФИНАЛЬНЫЙ КАТАЛОГ ПРАВИЛ КАЧЕСТВА ДАННЫХ (DQ CATALOG):")
display(df_dq.style.set_table_styles(style).hide(axis='index'))